# 04 — Build the supplied graphic, one perspective at a time

Use the supplied graphic to learn how a question, six thinking modes, Purple orchestration, and possible outcomes form a connected decision map. First rebuild its content; then compare an authored synthesis, make an edit, and export your practice.

**Suggested time:** 35–50 minutes. Run each cell with **Shift+Enter**, pausing at the reflection prompts, or use **Run → Run All Cells** for a complete demonstration. No assistant, API key, network request, new package, or existing decision file is needed. In JupyterLite, select **Python (Pyodide)** and keep the two helper modules and `examples/` folder beside this notebook.

**Three kinds of text:** **Source text** quotes/transcribes the graphic. **Teaching interpretation** explains or questions it. **Authored synthesis/practice** adds reasoning or edits; it is never attributed to the graphic. Edward de Bono developed the original six hats. Purple is this project's extension.

The notebook reads only bundled examples and writes separate snapshots to `graphic-exports/` and `example-exports/`. It never reads or writes personal `decision.json`. The personal workflow stays **01 → 02 → 03 → optional 06 → 05**.

The [reading lesson](https://jltobias.github.io/JupyterLite-Custom-GPT-Six-Thinking-Hats/book/graphic-lesson.html) and [full transcription](https://jltobias.github.io/JupyterLite-Custom-GPT-Six-Thinking-Hats/book/graphic-text.html) provide further discussion. All runtime content below is local.

## 1. See and read the original

**Source text and image.** The local PNG is an unchanged byte-for-byte copy of the supplied 2064 × 1024 image. Open `examples/plugin-output-example.png` in the file browser to zoom.

White, Red, and Black sit on the left; Yellow, Green, and Blue sit on the right. Purple spans the top. The globe names the subject and the bottom panel lists possible outcomes. Colored lines relate each perspective to the same subject; they do not establish causation or agreement.

The readable local JSON preserves the exact central question, all seven items and four questions in each original hat, Purple's role and six desired outputs, and all six possible outcomes. `purple.focus` holds the Role and `purple.items` holds the Key Outputs. Decorative callouts and the globe/hat artwork remain in the original PNG and full transcription; SVG cards rebuild content and connections, **not the original artwork**.

**Pause:** which statement looks like evidence but still needs a source? Both cells below can run independently after a restart.

In [ ]:
from pathlib import Path
from base64 import b64encode
from IPython.display import HTML, display

original_image = None
try:
    original_image = Path('examples/plugin-output-example.png').read_bytes()
    if not original_image.startswith(b'\x89PNG\r\n\x1a\n'):
        raise ValueError('The local image is not a PNG.')
    image_data = b64encode(original_image).decode('ascii')
    display(HTML('<figure><img style="max-width:100%;height:auto" '
                 'alt="Original graphic: a globe-centered global-health GIS question, six hat panels, Purple orchestration above, and possible outcomes below. The next cell provides all meaningful panel text." '
                 'src="data:image/png;base64,' + image_data + '">'
                 '<figcaption>Original supplied graphic, unchanged. Open the local PNG to zoom.</figcaption></figure>'))
except (OSError, ValueError) as error:
    original_image = None
    print('Action needed: restore examples/plugin-output-example.png from the repository, then rerun this cell.', error)

In [ ]:
from IPython.display import HTML, display
from decision_workflow import load_record, to_html, ValidationError

source_view = None
try:
    source_view = load_record('examples/graphic_source.json')
    display(HTML('<details><summary>Read all original panel text (local source reconstruction)</summary>'
                 + to_html(source_view, standalone=False, include_map=False) + '</details>'))
except (ValidationError, OSError) as error:
    source_view = None
    print('Action needed: restore examples/graphic_source.json from the repository, then rerun this cell.', error)

## 2. Begin with the central question

**Source text.** The original places this question inside Purple's panel. We make it the root card so every perspective has a common reference. `new_record` supplies six empty hat objects; empty Purple and outcomes placeholders mark what is still to come. `visible_sections=[]` shows the question card only. It labels the view as partial, rather than implying that missing perspectives are complete.

The setup cell also defines two small display/recovery functions. The actual content assignments remain visible in the later cells. Rerunning setup intentionally resets the in-memory lesson and practice; it does not change any files or earlier exports.

**Pause:** what makes this a broad professional-development question, rather than a decision about one health intervention?

In [ ]:
from copy import deepcopy
from html import escape
from IPython.display import HTML, display
from decision_workflow import HATS, load_record, new_record, validate_record, to_html, ValidationError
from six_hats_mindmap import render_mind_map

graphic_source = lesson = practice = None
visible_sections = None
lesson_svg = lesson_paths = synthesis_example = None

def check_lesson(required=()):
    """Check setup and the local source before presenting a current map/export."""
    if lesson is None or graphic_source is None or visible_sections is None:
        raise ValidationError('Run the central-question setup cell successfully first.')
    current_source = load_record('examples/graphic_source.json')
    if current_source != graphic_source:
        raise ValidationError('The source changed since setup. Restore canonical examples/graphic_source.json from the repository, then retry. Customize the lesson or practice assignments, not the source fixture.')
    validate_record(lesson)
    missing = [name for name in required if name not in visible_sections]
    if missing:
        raise ValidationError('Run these preceding steps first: ' + ', '.join(missing) + '.')

def show_stage(svg, caption, *, expanded=False):
    """Keep intermediate maps collapsed and usable at the page width."""
    responsive_svg = svg.replace('<svg ', '<svg style="max-width:100%;height:auto" ', 1)
    display(HTML('<details' + (' open' if expanded else '') + '><summary>' +
                 escape(caption) + '</summary>' + responsive_svg + '</details>'))

def label_lesson_edits(candidate, sections):
    """Identify edited panels without attributing learner text to the source."""
    edited = []
    marker = ' [Learner-authored edit]'
    for name in sections:
        if name != 'outcomes':
            panel = candidate[name]
            if panel.get('label', '').endswith(marker):
                panel['label'] = panel['label'][:-len(marker)]
        if candidate[name] != graphic_source[name]:
            edited.append(name.title())
            if name != 'outcomes':
                candidate[name]['label'] = candidate[name].get('label', name.title() + ' Hat') + marker
    candidate['demo_notice'] = (
        'LEARNER-AUTHORED MODIFICATIONS — edited displayed panels: ' + ', '.join(edited) +
        '. Other panels retain source wording. These edits are not statements from the original graphic or verified findings.'
        if edited else graphic_source['demo_notice'])
    candidate['context'] = (
        'Teaching base: the supplied graphic. Panels named in the notice include learner-authored edits; the bundled source fixture remains separate.'
        if edited else graphic_source['context'])

try:
    graphic_source = load_record('examples/graphic_source.json')
    if any(key not in graphic_source for key in ('subtitle', 'context', 'demo_notice', 'purple', 'outcomes')):
        raise ValidationError('The source fixture is incomplete; restore examples/graphic_source.json from the repository.')
    if 'label' not in graphic_source['purple']:
        raise ValidationError('The source fixture needs purple.label; restore examples/graphic_source.json from the repository.')
    lesson = new_record(graphic_source['title'], graphic_source['question'], graphic_source['context'])
    lesson['subtitle'] = graphic_source['subtitle']
    lesson['demo_notice'] = graphic_source['demo_notice']
    lesson['purple'] = {'label': graphic_source['purple']['label'], 'items': []}
    lesson['outcomes'] = []
    visible_sections = []
    lesson_svg = render_mind_map(lesson, visible_sections=visible_sections)
    show_stage(lesson_svg, 'Stage 0 — central question only', expanded=True)
except (ValidationError, OSError) as error:
    graphic_source = lesson = visible_sections = lesson_svg = None
    print('Action needed: restore examples/graphic_source.json from the repository, then rerun this setup cell.', error)

### Step 1 — White: Facts and evidence

**Teaching interpretation:** The panel inventories skills, sources, evidence categories, and gaps. A category such as “Evidence of impact” supplies no finding by itself.

**Source text by default:** the editable assignment below preserves this panel's seven items and four questions. Run it to add the panel to the map. A candidate copy is validated before replacing this hat; malformed edits leave earlier valid work intact. You can edit a quoted item, rerun the cell, and inspect the accumulating map. Modified panels are labeled learner-authored. The bundled source file is never modified. If you already made a practice copy, refresh it and reapply practice edits to include later hat changes.

**Pause:** Choose one category. What observation would turn it into usable evidence?

In [ ]:
from decision_workflow import ValidationError
lesson_svg = None
if not callable(globals().get('check_lesson')):
    print('Action needed: run the central-question setup cell, then the hat steps above this cell.')
else:
    try:
        check_lesson([])
        candidate = deepcopy(lesson)
        candidate['white'] = {
            "label": "White Hat — Facts & Evidence",
            "focus": "What do we know? What don't we know?",
            "items": [
                "Current skills and experience (GIS, spatial epidemiology, EO/remote sensing, climate & health, data science)",
                "Available data sources (EO, climate, health, mobility, etc.)",
                "Trends in AI, GeoAI, and geospatial technology",
                "Evidence of impact (publications, case studies, best practices)",
                "Assumptions, uncertainties, and information gaps",
                "Standards, tools, and infrastructure (open data, cloud, Jupyter)",
                "Policy, funding, and organizational context"
            ],
            "questions": [
                "What evidence supports different career paths?",
                "What data, skills, or resources are missing?",
                "What is the current state of GeoAI in global health?",
                "Where are the biggest knowledge gaps?"
            ]
        }
        validate_record(candidate)
        candidate_sections = visible_sections.copy()
        if 'white' not in candidate_sections:
            candidate_sections.append('white')
        label_lesson_edits(candidate, candidate_sections)
        candidate_svg = render_mind_map(candidate, visible_sections=candidate_sections)
        lesson, visible_sections, lesson_svg = candidate, candidate_sections, candidate_svg
        show_stage(lesson_svg, 'Stage 1 — through White (expand the accumulating map)')
    except (ValidationError, OSError) as error:
        lesson_svg = None
        print('Action needed:', error)
        print('Restore examples/graphic_source.json if needed, then rerun this cell. After a restart, run the central-question setup and preceding steps.')

### Step 2 — Red: Feelings and intuition

**Teaching interpretation:** Feelings, values, and personal motivations deserve explicit space. They are not measurements of a real person. The privacy concern here expresses a feeling; Black will examine a failure mode.

**Source text by default:** the editable assignment below preserves this panel's seven items and four questions. Run it to add the panel to the map. A candidate copy is validated before replacing this hat; malformed edits leave earlier valid work intact. You can edit a quoted item, rerun the cell, and inspect the accumulating map. Modified panels are labeled learner-authored. The bundled source file is never modified. If you already made a practice copy, refresh it and reapply practice edits to include later hat changes.

**Pause:** Whose feelings are missing from this professional-centered map?

In [ ]:
from decision_workflow import ValidationError
lesson_svg = None
if not callable(globals().get('check_lesson')):
    print('Action needed: run the central-question setup cell, then the hat steps above this cell.')
else:
    try:
        check_lesson([])
        candidate = deepcopy(lesson)
        candidate['red'] = {
            "label": "Red Hat — Feelings & Intuition",
            "focus": "What are my gut feelings? What matters to me?",
            "items": [
                "Excitement about new possibilities with GeoAI",
                "Passion for global health impact and equity",
                "Concerns about geoprivacy and protecting key populations (e.g., HIV)",
                "Desire to learn, build, and share (JupyterLite/JupyterBook)",
                "Frustration with slow adoption or technical barriers",
                "Values: ethics, inclusion, open science, real-world impact",
                "Intuition about which opportunities feel most meaningful"
            ],
            "questions": [
                "What energizes me most about this future?",
                "What worries me?",
                "What feels aligned with my values?",
                "What do I most want to contribute to the world?"
            ]
        }
        validate_record(candidate)
        candidate_sections = visible_sections.copy()
        if 'red' not in candidate_sections:
            candidate_sections.append('red')
        label_lesson_edits(candidate, candidate_sections)
        candidate_svg = render_mind_map(candidate, visible_sections=candidate_sections)
        lesson, visible_sections, lesson_svg = candidate, candidate_sections, candidate_svg
        show_stage(lesson_svg, 'Stage 2 — through Red (expand the accumulating map)')
    except (ValidationError, OSError) as error:
        lesson_svg = None
        print('Action needed:', error)
        print('Restore examples/graphic_source.json if needed, then rerun this cell. After a restart, run the central-question setup and preceding steps.')

### Step 3 — Black: Risks and challenges

**Teaching interpretation:** Identify how a plausible benefit might fail or harm someone. A risk statement should inform scope, checks, or stop conditions.

**Source text by default:** the editable assignment below preserves this panel's seven items and four questions. Run it to add the panel to the map. A candidate copy is validated before replacing this hat; malformed edits leave earlier valid work intact. You can edit a quoted item, rerun the cell, and inspect the accumulating map. Modified panels are labeled learner-authored. The bundled source file is never modified. If you already made a practice copy, refresh it and reapply practice edits to include later hat changes.

**Pause:** Connect re-identification risk to the Red privacy concern. What different work does each hat do?

In [ ]:
from decision_workflow import ValidationError
lesson_svg = None
if not callable(globals().get('check_lesson')):
    print('Action needed: run the central-question setup cell, then the hat steps above this cell.')
else:
    try:
        check_lesson([])
        candidate = deepcopy(lesson)
        candidate['black'] = {
            "label": "Black Hat — Risks & Challenges",
            "focus": "What could go wrong? What are the limitations?",
            "items": [
                "AI may automate parts of traditional GIS work",
                "Risk of spatial bias, false precision, and model misuse",
                "Geoprivacy and re-identification risks (especially for HIV and key populations)",
                "Data quality and underrepresentation of vulnerable regions",
                "Technical and computational constraints (e.g., JupyterLite limits)",
                "Ethical, legal, and data governance challenges",
                "Sustainability, funding, and career uncertainty"
            ],
            "questions": [
                "What are the biggest risks to my career and impact?",
                "Where could AI do harm?",
                "How might my assumptions be wrong?",
                "What constraints might limit success?"
            ]
        }
        validate_record(candidate)
        candidate_sections = visible_sections.copy()
        if 'black' not in candidate_sections:
            candidate_sections.append('black')
        label_lesson_edits(candidate, candidate_sections)
        candidate_svg = render_mind_map(candidate, visible_sections=candidate_sections)
        lesson, visible_sections, lesson_svg = candidate, candidate_sections, candidate_svg
        show_stage(lesson_svg, 'Stage 3 — through Black (expand the accumulating map)')
    except (ValidationError, OSError) as error:
        lesson_svg = None
        print('Action needed:', error)
        print('Restore examples/graphic_source.json if needed, then rerun this cell. After a restart, run the central-question setup and preceding steps.')

### Step 4 — Yellow: Benefits and opportunities

**Teaching interpretation:** Possible gains need reasons and eventual evidence. Pair the opportunity to teach with the White evidence gaps and Black limitations.

**Source text by default:** the editable assignment below preserves this panel's seven items and four questions. Run it to add the panel to the map. A candidate copy is validated before replacing this hat; malformed edits leave earlier valid work intact. You can edit a quoted item, rerun the cell, and inspect the accumulating map. Modified panels are labeled learner-authored. The bundled source file is never modified. If you already made a practice copy, refresh it and reapply practice edits to include later hat changes.

**Pause:** What would show that one proposed benefit had occurred?

In [ ]:
from decision_workflow import ValidationError
lesson_svg = None
if not callable(globals().get('check_lesson')):
    print('Action needed: run the central-question setup cell, then the hat steps above this cell.')
else:
    try:
        check_lesson([])
        candidate = deepcopy(lesson)
        candidate['yellow'] = {
            "label": "Yellow Hat — Benefits & Opportunities",
            "focus": "What are the positives? Why could this work?",
            "items": [
                "AI and GeoAI can amplify global health impact",
                "Integration of EO, climate, health, and mobility data",
                "New insights for early warning, targeting, and equity",
                "Growing demand for geospatial health intelligence",
                "Open-source tools and cloud platforms lower barriers",
                "Opportunity to educate and build global capacity",
                "Potential to shape responsible AI in global health"
            ],
            "questions": [
                "What are the biggest opportunities?",
                "What unique value can I offer?",
                "Where can AI make a real difference in global health?",
                "What would success look like?"
            ]
        }
        validate_record(candidate)
        candidate_sections = visible_sections.copy()
        if 'yellow' not in candidate_sections:
            candidate_sections.append('yellow')
        label_lesson_edits(candidate, candidate_sections)
        candidate_svg = render_mind_map(candidate, visible_sections=candidate_sections)
        lesson, visible_sections, lesson_svg = candidate, candidate_sections, candidate_svg
        show_stage(lesson_svg, 'Stage 4 — through Yellow (expand the accumulating map)')
    except (ValidationError, OSError) as error:
        lesson_svg = None
        print('Action needed:', error)
        print('Restore examples/graphic_source.json if needed, then rerun this cell. After a restart, run the central-question setup and preceding steps.')

### Step 5 — Green: Creativity and alternatives

**Teaching interpretation:** Ideas increase the available choices. A prototype is a learning experiment, not proof that a health system is ready for use.

**Source text by default:** the editable assignment below preserves this panel's seven items and four questions. Run it to add the panel to the map. A candidate copy is validated before replacing this hat; malformed edits leave earlier valid work intact. You can edit a quoted item, rerun the cell, and inspect the accumulating map. Modified panels are labeled learner-authored. The bundled source file is never modified. If you already made a practice copy, refresh it and reapply practice edits to include later hat changes.

**Pause:** Name a non-AI alternative to one idea. What could comparing them teach you?

In [ ]:
from decision_workflow import ValidationError
lesson_svg = None
if not callable(globals().get('check_lesson')):
    print('Action needed: run the central-question setup cell, then the hat steps above this cell.')
else:
    try:
        check_lesson([])
        candidate = deepcopy(lesson)
        candidate['green'] = {
            "label": "Green Hat — Creativity & Alternatives",
            "focus": "What else is possible? What are new ideas?",
            "items": [
                "JupyterLite prototypes for fail-fast learning and experimentation",
                "Interactive notebooks for EO + health + AI",
                "LLM-powered geospatial analysis assistants",
                "GeoAI for climate-health risk mapping",
                "AI red teaming and geoprivacy demos",
                "Six Thinking Hats AI agents for decision intelligence",
                "New roles: Geospatial Health Intelligence Scientist, Advisor, Educator, Open-Source Builder"
            ],
            "questions": [
                "What unconventional ideas could I try?",
                "How can I combine different data and methods?",
                "What experiments could I build in a Jupyter notebook?",
                "How can I learn in public and invite collaboration?"
            ]
        }
        validate_record(candidate)
        candidate_sections = visible_sections.copy()
        if 'green' not in candidate_sections:
            candidate_sections.append('green')
        label_lesson_edits(candidate, candidate_sections)
        candidate_svg = render_mind_map(candidate, visible_sections=candidate_sections)
        lesson, visible_sections, lesson_svg = candidate, candidate_sections, candidate_svg
        show_stage(lesson_svg, 'Stage 5 — through Green (expand the accumulating map)')
    except (ValidationError, OSError) as error:
        lesson_svg = None
        print('Action needed:', error)
        print('Restore examples/graphic_source.json if needed, then rerun this cell. After a restart, run the central-question setup and preceding steps.')

### Step 6 — Blue: Process and next steps

**Teaching interpretation:** Goals, scope, measures, collaborators, and review points organize the thinking. Blue manages a process; Purple will integrate what the perspectives imply.

**Source text by default:** the editable assignment below preserves this panel's seven items and four questions. Run it to add the panel to the map. A candidate copy is validated before replacing this hat; malformed edits leave earlier valid work intact. You can edit a quoted item, rerun the cell, and inspect the accumulating map. Modified panels are labeled learner-authored. The bundled source file is never modified. If you already made a practice copy, refresh it and reapply practice edits to include later hat changes.

**Pause:** Which action needs an owner and a review date before it becomes usable?

In [ ]:
from decision_workflow import ValidationError
lesson_svg = None
if not callable(globals().get('check_lesson')):
    print('Action needed: run the central-question setup cell, then the hat steps above this cell.')
else:
    try:
        check_lesson([])
        candidate = deepcopy(lesson)
        candidate['blue'] = {
            "label": "Blue Hat — Process & Next Steps",
            "focus": "How do we organize and move forward?",
            "items": [
                "Define goals (skills, projects, career outcomes)",
                "Set scope and priorities for learning and experimentation",
                "Choose metrics for success (e.g., impact, publications, tools)",
                "Develop a 12-month roadmap (learn → build → iterate → teach)",
                "Create a JupyterBook and JupyterLite repository structure",
                "Engage collaborators and open-source community",
                "Regularly review progress and adapt"
            ],
            "questions": [
                "What are my top 3–5 goals for the next 1–3 years?",
                "What should I do first?",
                "How will I measure progress?",
                "What support or collaborations do I need?"
            ]
        }
        validate_record(candidate)
        candidate_sections = visible_sections.copy()
        if 'blue' not in candidate_sections:
            candidate_sections.append('blue')
        label_lesson_edits(candidate, candidate_sections)
        candidate_svg = render_mind_map(candidate, visible_sections=candidate_sections)
        lesson, visible_sections, lesson_svg = candidate, candidate_sections, candidate_svg
        show_stage(lesson_svg, 'Stage 6 — through Blue (expand the accumulating map)')
    except (ValidationError, OSError) as error:
        lesson_svg = None
        print('Action needed:', error)
        print('Restore examples/graphic_source.json if needed, then rerun this cell. After a restart, run the central-question setup and preceding steps.')

## 3. Read the connections

**Teaching interpretation.** Expand the Blue stage to inspect all six panels. The SVG arranges White/Yellow, Red/Green, and Black/Blue around a central spine. Each visible panel has a connector; hidden panels have none. These connectors mean “contributes to this decision,” not “proves” or “causes.” Canonical position stays stable even if you rerun cells in a different order.

| Connection to reason about | Why it matters |
|---|---|
| Yellow education opportunity + Green JupyterLite prototype | A possible teaching experiment. |
| Red open-science values + Black geoprivacy risk | Sharing and protecting people can create a tension. |
| White information gaps + Blue success measures | Decide what to learn and how to review it. |

**Pause:** trace a second connection using two actual items. Which stakeholder or assumption is absent? Drawing a line does not perform that reasoning for you.

## 4. Add Purple's role and desired outputs

**Source text:** Purple coordinates the six modes. Its `focus` below is the graphic's Role, and its six `items` are the Key Outputs. The graphic asks for combined insights, tradeoffs, blind spots, a roadmap, learning ideas, and next steps; it does **not** supply a completed recommendation. No `author` or `recommendation` is invented for this reconstruction.

**Teaching interpretation:** Purple is the project's extension to Edward de Bono's six hats. It synthesizes perspectives; Blue organizes the thinking process. **Pause:** which of these desired outputs would require additional reasoning or evidence?

In [ ]:
from decision_workflow import ValidationError
lesson_svg = None
if not callable(globals().get('check_lesson')):
    print('Action needed: run the central-question setup cell, then the hat steps above this cell.')
else:
    try:
        check_lesson(['white', 'red', 'black', 'yellow', 'green', 'blue'])
        candidate = deepcopy(lesson)
        candidate['purple'] = {
            "label": "Purple Hat — Orchestrator",
            "focus": "Coordinate the six thinking modes, identify connections, surface blind spots, and create an integrated pathway for your evolution as a global health GIS professional in the age of AI.",
            "items": [
                "Combined insights from all hats",
                "Key themes, tensions, and tradeoffs",
                "Blind spots and the “elephant in the room”",
                "Integrated recommendations and roadmap",
                "Ideas for JupyterLite/JupyterBook learning",
                "Next steps for action"
            ]
        }
        validate_record(candidate)
        candidate_sections = visible_sections.copy()
        if 'purple' not in candidate_sections:
            candidate_sections.append('purple')
        label_lesson_edits(candidate, candidate_sections)
        candidate_svg = render_mind_map(candidate, visible_sections=candidate_sections)
        lesson, visible_sections, lesson_svg = candidate, candidate_sections, candidate_svg
        show_stage(lesson_svg, 'Stage 7 — six perspectives plus Purple’s role')
    except (ValidationError, OSError) as error:
        lesson_svg = None
        print('Action needed:', error)
        print('Restore examples/graphic_source.json if needed, then rerun this cell. After a restart, run the central-question setup and preceding steps.')

## 5. Add possible outcomes and inspect the reconstruction

**Source text:** the six outcomes below are aspirations, not observed results. The final map includes every original panel's meaningful text. Purple's tagline is preserved in the source context. Open the complete map, then use the text view if the long SVG is difficult to read.

**Pause:** which outcome could a short notebook exercise plausibly assess, and which needs a longer evaluation?

In [ ]:
from decision_workflow import ValidationError
lesson_svg = None
if not callable(globals().get('check_lesson')):
    print('Action needed: run the central-question setup cell, then the hat steps above this cell.')
else:
    try:
        check_lesson(['white', 'red', 'black', 'yellow', 'green', 'blue', 'purple'])
        candidate = deepcopy(lesson)
        candidate['outcomes'] = [
            "Enhanced skills and professional growth",
            "Innovative GeoAI prototypes and tools",
            "JupyterBook and JupyterLite learning ecosystem",
            "Stronger global network and collaborations",
            "Greater impact on global health equity",
            "A responsible, ethical, and sustainable future"
        ]
        validate_record(candidate)
        candidate_sections = visible_sections.copy()
        if 'outcomes' not in candidate_sections:
            candidate_sections.append('outcomes')
        label_lesson_edits(candidate, candidate_sections)
        candidate_svg = render_mind_map(candidate, visible_sections=candidate_sections)
        lesson, visible_sections, lesson_svg = candidate, candidate_sections, candidate_svg
        show_stage(lesson_svg, 'Stage 8 — complete editable reconstruction (open final map)')
        display(HTML('<details><summary>Read the complete reconstruction as text</summary>'
                     + to_html(lesson, standalone=False, include_map=False) + '</details>'))
        print('Complete reconstruction ready. Source wording is preserved unless you edited a hat assignment.')
    except (ValidationError, OSError) as error:
        lesson_svg = None
        print('Action needed:', error)
        print('Restore examples/graphic_source.json if needed, then rerun this cell. After a restart, run the central-question setup and preceding steps.')

## 6. Compare with a worked authored synthesis

**Authored synthesis — teaching addition.** Now copy the reconstruction and replace Purple in that copy with a conditional recommendation. This example narrows the broad question to a first teaching trial. It draws on the reading lesson's two-week proposal. The original source and your reconstructed `lesson` keep Purple's role and desired outputs.

The reasoning connects Yellow's education opportunity, Green's notebooks, Red's wish to share responsibly, White's missing evidence, Black's risk of misleading maps, and Blue's review process. Conditions for changing course make the recommendation revisable.

**Proposed comparison rule (not a validated evaluation):** after each explanation, award one point for each correctly explained exit-ticket element: an unsupported claim, an evidence gap, a limitation, and a condition that changes the recommendation (0–4 points). Compare the same learners' mean scores for the two versions using the same questions and scoring criteria. Continue the AI version only if its mean is higher and it introduces no new misconceptions. A tie favors the simpler non-AI explanation; any new misconception prompts revision before continuing.

**Pause:** point to a statement from each hat that supports or limits this choice. What evidence could overturn it?

In [ ]:
from decision_workflow import ValidationError
lesson_svg = None
if not callable(globals().get('check_lesson')):
    print('Action needed: run the central-question setup cell, then the hat steps above this cell.')
else:
    try:
        check_lesson(['white', 'red', 'black', 'yellow', 'green', 'blue', 'purple', 'outcomes'])
        candidate = deepcopy(lesson)
        candidate['demo_notice'] = 'AUTHORED SYNTHESIS COMPARISON — six panels are copied from the current lesson and may include learner-authored edits. Purple below is a lesson-authored proposal, not source text or verified findings. Recheck its reasoning against any edited panels.'
        candidate['purple'] = {
            "label": "Purple — worked authored synthesis (teaching addition)",
            "focus": "A possible first teaching trial; not a conclusion present in the source graphic.",
            "author": "Lesson-authored example for discussion; requires learner and stakeholder review.",
            "items": [
                "Yellow’s education opportunity and Green’s JupyterLite experiments suggest a small teaching trial, aligned with Red’s wish to learn and share responsibly.",
                "White’s information gaps and Black’s false-precision and privacy risks limit what a convincing map can establish; Blue’s review process makes the trial revisable."
            ],
            "themes": [
                "Learn, build, share, and review with intended learners."
            ],
            "tensions": [
                "Moving quickly conflicts with demonstrating useful learning without misconceptions."
            ],
            "blind_spots": [
                "Intended learners and affected communities have not contributed their priorities."
            ],
            "recommendation": "Try a two-week teaching exercise using fictional district-level counts. Compare an AI-assisted explanation with a simple non-AI explanation. Continue the AI version only if it improves understanding without introducing new misconceptions.",
            "uncertainties": [
                "Whether AI adds learning value is unknown. Fictional counts do not establish real disease risk."
            ],
            "decision_pivots": [
                "If the simple explanation works as well, retain it. If learners acquire new misconceptions, revise the exercise before continuing."
            ],
            "next_steps": [
                "The lesson author first asks intended learners what they need.",
                "The author and a peer reviewer review on day 14 after the agreed start date, entered on the calendar at planning time.",
                "Compare a four-item exit ticket: an unsupported claim, an evidence gap, a limitation, and a condition that changes a recommendation. Record new misconceptions. These are proposed criteria, not validated results."
            ]
        }
        validate_record(candidate)
        candidate_svg = render_mind_map(candidate, visible_sections=['purple'])
        synthesis_example, lesson_svg = candidate, candidate_svg
        show_stage(lesson_svg, 'Authored Purple comparison — expand the proposed reasoning')
        display(HTML('<details><summary>Read the authored synthesis and its copied perspectives as text</summary>'
                     + to_html(synthesis_example, standalone=False, include_map=False) + '</details>'))
    except (ValidationError, OSError) as error:
        lesson_svg = None
        print('Action needed:', error)
        print('Restore examples/graphic_source.json if needed, then rerun this cell. After a restart, run the central-question setup and preceding steps.')

## 7. Compare original, reconstruction, and fictional adaptation

| Object | What it contains |
|---|---|
| `examples/plugin-output-example.png` | The unchanged supplied artwork and source wording. |
| `examples/graphic_source.json` and reconstructed `lesson` | Faithful panel content; SVG cards make it editable and connected. Purple is a role and desired outputs. |
| `synthesis_example` above | An explicitly authored two-week teaching proposal added to a separate copy. |
| `examples/global_health.json` below | The existing fictional adaptation, with rewritten statements, candidate-source notes, and an authored four-week prototype recommendation. |

The next cell can run independently after a restart. Its collapsed map and reading view show the **adaptation**. It does not replace the source reconstruction. **Pause:** find one retained idea, one rewritten claim, and one added decision pivot. Why might two authored proposals choose different time horizons?

In [ ]:
from IPython.display import HTML, display
from decision_workflow import load_record, to_html, ValidationError
from six_hats_mindmap import render_mind_map

example = None
try:
    example = load_record('examples/global_health.json')
    print(example['demo_notice'])
    adaptation_svg = render_mind_map(example).replace('<svg ', '<svg style="max-width:100%;height:auto" ', 1)
    display(HTML('<details><summary>Inspect the separate fictional adaptation map</summary>' + adaptation_svg + '</details>'))
    display(HTML('<details><summary>Read the fictional adaptation as text</summary>'
                 + to_html(example, standalone=False, include_map=False) + '</details>'))
except (ValidationError, OSError) as error:
    example = None
    print('Action needed: the example file could not be read.', error)
    print('Restore examples/global_health.json from the repository beside these notebooks, then rerun this cell.')

## 8. Change one item and rerender

**Authored practice.** Make a separate working copy of your reconstruction, then run the edit cell. Its default edit turns Green's first broad idea into a small, concrete learning experiment. Change the quoted text to your own idea and rerun **only the edit, rerender, and lesson-export cells**. The map and new exported JSON will contain your edit. A list index of `0` selects the first item.

To undo all practice edits, rerun the practice-copy cell. To restore a changed hat assignment, copy its source wording from `graphic_source[hat]` or reopen the clean notebook. Rerunning the central-question setup resets the construction; rerun the following stages afterward. Kernel restart also clears unsaved in-memory changes.

**After a later reconstruction edit:** `practice` remains an independent copy. Save any practice text you want to keep, rerun the practice-copy cell to refresh from the current lesson, then reapply your practice edits and rerun the map and export cells. Refreshing replaces the in-memory practice copy; nothing is automatically merged or overwritten in saved snapshots.

In [ ]:
from decision_workflow import ValidationError
lesson_svg = None
if not callable(globals().get('check_lesson')):
    print('Action needed: run the central-question setup cell, then the hat steps above this cell.')
else:
    try:
        check_lesson(['white', 'red', 'black', 'yellow', 'green', 'blue', 'purple', 'outcomes'])
        candidate = deepcopy(lesson)
        candidate['demo_notice'] = 'AUTHORED PRACTICE — copied from the current lesson, which may already include learner-authored edits. Further practice changes belong to the learner, not the original graphic. This is not a verified assessment or a completed synthesis.'
        validate_record(candidate)
        practice = candidate
        print('Practice copy ready. Run the next cell to make the demonstrated edit.')
    except (ValidationError, OSError) as error:
        lesson_svg = None
        print('Action needed:', error)
        print('Restore examples/graphic_source.json if needed, then rerun this cell. After a restart, run the central-question setup and preceding steps.')

In [ ]:
from decision_workflow import ValidationError
lesson_svg = None
if not callable(globals().get('check_lesson')):
    print('Action needed: run the central-question setup cell, then the hat steps above this cell.')
else:
    try:
        check_lesson(['white', 'red', 'black', 'yellow', 'green', 'blue', 'purple', 'outcomes'])
        if globals().get('practice') is None:
            raise ValidationError('Run the practice-copy cell first.')
        validate_record(practice)
        if not practice['green']['items']:
            raise ValidationError('Green has no first item. Run the practice-copy cell to restore the starting items.')
        candidate = deepcopy(practice)
        candidate['green']['items'][0] = 'Learner exercise: build a tiny JupyterLite comparison of AI and non-AI explanations using fictional district counts.'
        validate_record(candidate)
        practice = candidate
        print('Practice edit applied to Green item 1. Run the rerender and lesson-export cells next.')
    except (ValidationError, OSError) as error:
        lesson_svg = None
        print('Action needed:', error)
        print('Restore examples/graphic_source.json if needed, then rerun this cell. After a restart, run the central-question setup and preceding steps.')

In [ ]:
from decision_workflow import ValidationError
lesson_svg = None
if not callable(globals().get('check_lesson')):
    print('Action needed: run the central-question setup cell, then the hat steps above this cell.')
else:
    try:
        check_lesson(['white', 'red', 'black', 'yellow', 'green', 'blue', 'purple', 'outcomes'])
        if globals().get('practice') is None:
            raise ValidationError('Run the practice-copy and practice-edit cells first.')
        validate_record(practice)
        lesson_svg = render_mind_map(practice)
        show_stage(lesson_svg, 'Your edited complete practice map (open to inspect Green item 1)')
        print('Current Green item 1:', practice['green']['items'][0] if practice['green']['items'] else '(no entries)')
    except (ValidationError, OSError) as error:
        lesson_svg = None
        print('Action needed:', error)
        print('Restore examples/graphic_source.json if needed, then rerun this cell. After a restart, run the central-question setup and preceding steps.')

## 9. Export your lesson practice

Run this after an edit to save JSON, SVG, Markdown, and accessible HTML in a new snapshot folder under `graphic-exports/`. Each run creates a unique folder; earlier snapshots stay unchanged. Download the files using the links or the file browser. The exported HTML has headings and lists plus an expandable SVG. JSON preserves your editable record.

**Check:** open the downloaded JSON or HTML and locate your changed Green item. This snapshot is the **authored practice**, not a new version of the original source. The independently runnable adapted-example export below remains available for comparison.

In [ ]:
from IPython.display import HTML, display
from decision_workflow import export_record, download_links, ValidationError

lesson_paths = None
try:
    if not callable(globals().get('check_lesson')):
        raise ValidationError('Run the central-question setup, hat steps, and practice-copy/edit cells first.')
    check_lesson(['white', 'red', 'black', 'yellow', 'green', 'blue', 'purple', 'outcomes'])
    if globals().get('practice') is None:
        raise ValidationError('Run the practice-copy and practice-edit cells first.')
    lesson_paths = export_record(practice, 'graphic-exports', stem='graphic-practice')
except (ValidationError, OSError) as error:
    lesson_paths = None
    print('Action needed: lesson export did not finish.', error)
    print('Restore examples/graphic_source.json or complete the setup/practice steps as indicated; check available browser storage, then rerun this cell.')
else:
    print('Lesson snapshot folder:', lesson_paths['json'].parent)
    try:
        display(HTML(download_links(lesson_paths)))
    except OSError as error:
        print('Files saved, but embedded download links could not be prepared:', error)
        print('Use the file browser to open the snapshot folder above, right-click each file, and choose Download.')

### Separate adapted-example export

This cell reloads **`examples/global_health.json`**, even after a kernel restart, and saves the unchanged fictional adaptation to `example-exports/`. It does not use your practice edits. Compare its four-week authored Purple proposal with the source role and the two-week teaching proposal above.

In [ ]:
from IPython.display import HTML, display
from decision_workflow import load_record, export_record, download_links, ValidationError

example = None
example_paths = None
try:
    example = load_record('examples/global_health.json')
    example_paths = export_record(example, 'example-exports', stem='global-health-example')
except (ValidationError, OSError) as error:
    example = None
    example_paths = None
    print('Action needed: example export did not finish.', error)
    print('Check examples/global_health.json and available browser storage, then rerun this cell. Your decision.json is separate and has not been changed.')
else:
    print('Example snapshot folder:', example_paths['json'].parent)
    try:
        display(HTML(download_links(example_paths)))
    except OSError as error:
        print('Files saved, but embedded download links could not be prepared:', error)
        print('Use the file browser to open the snapshot folder above, right-click each file, and choose Download.')

## Explain and transfer

1. Name one source statement, one teaching interpretation, and one authored synthesis statement. How can a reader tell them apart?
2. Explain a connection and a tension between two hats. A connector alone is not an argument.
3. State what evidence would change your preferred next step, and one outcome this exercise has not demonstrated.

Return to the [guided lesson](https://jltobias.github.io/JupyterLite-Custom-GPT-Six-Thinking-Hats/book/graphic-lesson.html) for its five-part decision brief and rubric. For your own decision, follow **02 → 03 → 06 → 05** and include Notebook 06 to save an authored Purple recommendation. Keep the source, fictional adaptation, and personal decision distinct.